# Prepare Training Data

**Next:** [02-submit-training-job.ipynb](02-submit-training-job.ipynb)

---

This notebook validates, preprocesses, and uploads training data for fine-tuning small language models on Azure ML.

In [ ]:
# Import required modules
from pathlib import Path
import sys

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

from src.data.preprocessing import (
    validate_jsonl_format,
    load_jsonl_data,
    split_train_validation,
    calculate_data_statistics,
    save_jsonl_data,
)
from src.data.upload_to_blob import (
    upload_file_to_blob,
    upload_directory_to_blob,
)
from src.utils.config import load_config
from src.utils.logging_config import setup_logging

# Setup logging
logger = setup_logging(log_level="INFO")

## 1. Load Configuration

In [ ]:
# Load configuration
config = load_config()

print(f"Azure Subscription: {config.azure.subscription_id}")
print(f"Resource Group: {config.azure.resource_group}")
print(f"Storage Account: {config.storage.account_name}")
print(f"Container: {config.storage.container_name}")

## 2. Validate Training Data

In [ ]:
# Path to your training data (default location)
input_file = Path("../data/training_data.jsonl")

# Check if file exists
if not input_file.exists():
    print(f"❌ Training data not found at: {input_file}")
    print("\n💡 Quick fix options:")
    print("1. Place your training data at: data/training_data.jsonl")
    print("2. Use the example: cp ../data/training_data.jsonl.example ../data/training_data.jsonl")
    print("3. Update the path above to point to your JSONL file")
    raise FileNotFoundError(f"Training data not found: {input_file}")

print(f"📂 Using training data: {input_file}")

# Validate the format
is_valid, errors = validate_jsonl_format(input_file)

if is_valid:
    print("✅ Data validation successful!")
else:
    print("❌ Data validation failed:")
    for error in errors[:10]:  # Show first 10 errors
        print(f"  - {error}")
    if len(errors) > 10:
        print(f"  ... and {len(errors) - 10} more errors")
    raise ValueError("Data validation failed. Please fix the errors above.")

## 3. Load and Inspect Data

In [ ]:
# Load the data
data = load_jsonl_data(input_file)

print(f"Loaded {len(data)} samples")
print("\nFirst sample:")
print(f"Prompt: {data[0]['prompt'][:100]}...")
print(f"Completion: {data[0]['completion'][:100]}...")

## 4. Calculate Statistics

In [ ]:
# Calculate dataset statistics
stats = calculate_data_statistics(data)

print("Dataset Statistics:")
print(f"  Number of samples: {stats['num_samples']}")
print(f"  Avg prompt length: {stats['avg_prompt_length']:.1f} chars")
print(f"  Avg completion length: {stats['avg_completion_length']:.1f} chars")
print(f"  Total tokens (estimate): {stats['total_tokens_estimate']:,}")

## 5. Split into Train/Validation Sets

In [ ]:
# Split the data
train_data, val_data = split_train_validation(
    data,
    validation_split=0.2,
    seed=42
)

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

# Calculate stats for each split
train_stats = calculate_data_statistics(train_data)
val_stats = calculate_data_statistics(val_data)

## 6. Save Split Data Locally

In [ ]:
# Create output directory
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save train and validation files
train_file = output_dir / "train.jsonl"
val_file = output_dir / "val.jsonl"

save_jsonl_data(train_data, train_file)
save_jsonl_data(val_data, val_file)

print(f"✅ Saved training data to {train_file}")
print(f"✅ Saved validation data to {val_file}")

## 7. Upload to Azure Blob Storage

**Why upload to Azure Blob Storage?** Training jobs run on remote Azure ML compute, which can't access your local files. By uploading data to blob storage, Azure ML can securely mount and access it during training.

In [ ]:
# Upload train and validation files to blob storage
# Note: The container is named 'training-data', so we upload to the root
# This creates: training-data/train.jsonl (not training-data/training-data/train.jsonl)

print(f"📤 Uploading to storage account: {config.storage.account_name}")
print(f"   Container: {config.storage.container_name}")
print()

train_url = upload_file_to_blob(
    train_file,
    "train.jsonl",  # Upload to container root
    config.storage,
    overwrite=True
)

val_url = upload_file_to_blob(
    val_file,
    "val.jsonl",  # Upload to container root
    config.storage,
    overwrite=True
)

print(f"\n✅ Training data uploaded to: {train_url}")
print(f"✅ Validation data uploaded to: {val_url}")
print(f"\n💡 Files are in container '{config.storage.container_name}' at:")
print(f"   - train.jsonl")
print(f"   - val.jsonl")

In [ ]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import AzureBlobDatastore
from azure.identity import DefaultAzureCredential

# Initialize Azure ML client
credential = DefaultAzureCredential()
ml_client = MLClient(
    credential=credential,
    subscription_id=config.azure.subscription_id,
    resource_group_name=config.azure.resource_group,
    workspace_name=config.azure.workspace_name,
)

print(f"✓ Connected to Azure ML workspace: {config.azure.workspace_name}")

# Check if a datastore for our storage account already exists
datastore_name = "training_data_store"

try:
    # Try to get existing datastore
    existing_datastore = ml_client.datastores.get(datastore_name)
    print(f"✓ Datastore '{datastore_name}' already exists")
    print(f"  Account: {existing_datastore.account_name}")
    print(f"  Container: {existing_datastore.container_name}")
except Exception:
    # Create new datastore if it doesn't exist
    print(f"Creating new datastore '{datastore_name}'...")

    # Create datastore with credential-based authentication
    datastore = AzureBlobDatastore(
        name=datastore_name,
        description="Training data storage",
        account_name=config.storage.account_name,
        container_name=config.storage.container_name,
    )

    try:
        created_datastore = ml_client.datastores.create_or_update(datastore)
        print(f"✓ Created datastore '{datastore_name}'")
        print(f"  Account: {created_datastore.account_name}")
        print(f"  Container: {created_datastore.container_name}")
    except Exception as e:
        print(f"⚠️  Could not create datastore: {e}")
        print(f"\n💡 This may happen if:")
        print(f"   - You don't have permission to create datastores")
        print(f"   - The storage account requires additional authentication")
        print(f"   - The datastore already exists with a different configuration")
        print(f"\n   You can still proceed - notebook 02 will use the default datastore")

# Show the path that will be used in training
print(f"\n📁 Data paths for training job:")
print(f"  Datastore: {datastore_name}")
print(f"  Container: {config.storage.container_name}")
print(f"  Train file: azureml://datastores/{datastore_name}/paths/train.jsonl")
print(f"  Val file: azureml://datastores/{datastore_name}/paths/val.jsonl")

In [ ]:
from azure.storage.blob import BlobServiceClient

# Connect to blob storage
blob_service_client = BlobServiceClient(
    account_url=f"https://{config.storage.account_name}.blob.core.windows.net",
    credential=credential
)

container_client = blob_service_client.get_container_client(config.storage.container_name)

print(f"📦 Checking container: {config.storage.container_name}")
print(f"   Account: {config.storage.account_name}\n")

# List all blobs in the container
blobs = list(container_client.list_blobs())

if blobs:
    print(f"✓ Found {len(blobs)} files:\n")

    # Check for our specific training files
    expected_files = ["train.jsonl", "val.jsonl"]
    found_files = [blob.name for blob in blobs]

    for blob in blobs:
        size_mb = blob.size / (1024 * 1024)
        is_training_file = blob.name in expected_files
        marker = "✓" if is_training_file else " "
        print(f"  {marker} 📄 {blob.name}")
        print(f"       Size: {size_mb:.2f} MB")
        print(f"       Modified: {blob.last_modified}")
        print()

    # Check if training files exist
    missing_files = [f for f in expected_files if f not in found_files]

    if not missing_files:
        print("✅ All training data files found and accessible!")
    else:
        print(f"⚠️  Missing files: {', '.join(missing_files)}")
        print(f"   Found files may have wrong paths. Expected:")
        print(f"   - train.jsonl")
        print(f"   - val.jsonl")
else:
    print("❌ No files found in container")
    print("   The upload may have failed. Check the previous cell for errors.")

## 9. Verify Data in Blob Storage

**Why verify?** This step confirms that files were uploaded correctly and are accessible. It lists all files in your container and checks that the training files exist with the expected names and sizes.

## 8. Register Data with Azure ML Datastore

**Why register a datastore?** Azure ML uses datastores to reference external storage. By registering your blob storage container as a datastore, you create a reusable reference that training jobs can use to access the data without hardcoding credentials.

## Summary

✅ **Training data successfully prepared and uploaded!**

**What was completed:**
1. ✓ Data validated and split into train/validation sets
2. ✓ Files uploaded to Azure Blob Storage (container root)
3. ✓ Datastore registered with Azure ML workspace
4. ✓ Data verified and accessible

**Data location:**
- Storage account: As configured in your `.env` file
- Container: `training-data` (or as specified in AZURE_STORAGE_CONTAINER_NAME)
- Files: `train.jsonl` and `val.jsonl` (at container root)
- Azure ML Datastore: `training_data_store`
- Azure ML paths: `azureml://datastores/training_data_store/paths/train.jsonl`

Your data is now ready for remote training on Azure ML!

---

## Troubleshooting

**If data upload failed:**
- Check Azure credentials: `az account show`
- Verify storage account name in `.env`
- Ensure you have Storage Blob Data Contributor role on the storage account

**If datastore registration failed:**
- Check you have Azure ML Workspace Contributor role
- The training job can still use the default datastore as fallback
- Verify storage account is accessible from Azure ML workspace

**If verification shows wrong file paths:**
- Files should be at container root: `train.jsonl`, `val.jsonl`
- Not in a subfolder: `training-data/train.jsonl` ❌
- Re-run cell 8 to upload with correct paths

---

## Navigation

**Next:** [02-submit-training-job.ipynb](02-submit-training-job.ipynb)